# pHash Evasion — Attack Visualization (T=0.10, exactly 8 bits flipped)

**Author:** Avijit Roy — FDEPH Project  

Shows the minimum-cost evasion: images where the attack flipped exactly 8 bits  
(d_norm = 0.125, the first even-integer jump above T=0.10).

383 / 500 images (76.6%) achieved evasion at exactly 8 bits — these are shown here.

Each panel row shows:
- **Original image** — unmodified
- **Adversarial image** — looks identical to human observers  
- **Difference × 10** — perturbation amplified to make it visible
- **Original hash grid** — 64-bit pHash as 8×8 grid (blue=1, gray=0)
- **Adversarial hash grid** — same, with flipped bits highlighted in red
- **XOR grid** — only the 8 flipped bits shown in red


In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
import csv
import glob
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from utils.phash_torch import compute_phash_hard
from utils.image_processing import load_and_preprocess_img

ADV_DIR   = os.path.join(REPO_ROOT, "evasion_attack_outputs_phash")
ORI_DIR   = os.path.join(REPO_ROOT, "inputs", "inputs_500")
FIG_DIR   = os.path.join(REPO_ROOT, "fdeph_eval", "analysis", "figures")
LOG_T010  = os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.10.csv")
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device("cpu")
print("Setup complete.")

In [ ]:
# Load image IDs that flipped exactly 8 bits at T=0.10 (minimum-cost evasion)
ids_8bit = []
with open(LOG_T010) as f:
    for row in csv.DictReader(f):
        if row['success'] == '1' and float(row.get('dist_raw', 0)) == 8.0:
            ids_8bit.append({
                'image_id':   row['image_id'],
                'step':       int(row['step']),
                'elapsed_ms': float(row['elapsed_ms']),
                'ssim':       float(row['ssim']),
                'linf':       float(row['linf']),
                'l2':         float(row['l2']),
            })

print(f"Images with exactly 8 bits flipped at T=0.10: {len(ids_8bit)} / 500")
print(f"That is {100*len(ids_8bit)/500:.1f}% of all images")

# Sort by step count (fastest first) to pick interesting examples
ids_8bit.sort(key=lambda x: x['step'])
print(f"\nFastest 5:")
for r in ids_8bit[:5]:
    print(f"  {r['image_id']}  steps={r['step']}  time={r['elapsed_ms']:.0f}ms  "
          f"SSIM={r['ssim']:.4f}  L∞={r['linf']:.4f}")

In [ ]:
def find_original(image_id):
    """Find original image file by image_id stem."""
    for ext in ['.JPEG', '.jpeg', '.jpg', '.png', '.PNG']:
        p = os.path.join(ORI_DIR, image_id + ext)
        if os.path.exists(p):
            return p
    return None

def find_adversarial(image_id):
    p = os.path.join(ADV_DIR, image_id + '_opt.png')
    return p if os.path.exists(p) else None

# Build confirmed pairs (both files exist)
confirmed_pairs = []
for r in ids_8bit:
    ori = find_original(r['image_id'])
    adv = find_adversarial(r['image_id'])
    if ori and adv:
        confirmed_pairs.append({**r, 'ori_path': ori, 'adv_path': adv})

print(f"Confirmed pairs with both files present: {len(confirmed_pairs)}")

In [ ]:
def draw_hash_grid(ax, grid, title, flip_mask=None):
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 8)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=8, pad=3)
    for r in range(8):
        for c in range(8):
            bit = grid[r, c]
            if flip_mask is not None and flip_mask[r, c]:
                color, edge, lw = '#E24B4A', '#A32D2D', 1.2
            elif bit == 1:
                color, edge, lw = '#378ADD', '#185FA5', 0.4
            else:
                color, edge, lw = '#f0f0f0', '#cccccc', 0.4
            ax.add_patch(patches.Rectangle(
                (c, 7 - r), 1, 1, linewidth=lw, edgecolor=edge, facecolor=color
            ))

def render_row(ax_row, pair, amp=10):
    """Render one full row: 3 image panels + 3 hash panels."""
    ori_np = np.array(Image.open(pair['ori_path']).convert('RGB')).astype(float) / 255.0
    adv_pil = Image.open(pair['adv_path']).convert('RGB')

    # Resize adv to match original dimensions
    if adv_pil.size != (ori_np.shape[1], ori_np.shape[0]):
        adv_pil = adv_pil.resize((ori_np.shape[1], ori_np.shape[0]), Image.LANCZOS)
    adv_np = np.array(adv_pil).astype(float) / 255.0

    diff_np = np.clip(np.abs(ori_np - adv_np) * amp, 0, 1)

    # Compute hashes
    ori_t = load_and_preprocess_img(pair['ori_path'], device, resize=True)
    adv_t = load_and_preprocess_img(pair['adv_path'], device, resize=True)
    ori_hash, ori_hex, _ = compute_phash_hard(ori_t)
    adv_hash, adv_hex, _ = compute_phash_hard(adv_t)

    flip_mask = (ori_hash != adv_hash).numpy().reshape(8, 8)
    bits_flipped = int(flip_mask.sum())
    d_norm = bits_flipped / 64.0

    # Image panels
    ax_row[0].imshow(ori_np)
    ax_row[0].axis('off')

    ax_row[1].imshow(adv_np)
    ax_row[1].axis('off')
    ax_row[1].set_title(
        f"step {pair['step']}  {pair['elapsed_ms']:.0f}ms\n"
        f"SSIM={pair['ssim']:.4f}  L∞={pair['linf']:.4f}",
        fontsize=7, color='#444'
    )

    ax_row[2].imshow(diff_np)
    ax_row[2].axis('off')
    ax_row[2].set_title(f"Difference ×{amp}\nL∞={pair['linf']:.4f}", fontsize=7)

    # Hash grids
    draw_hash_grid(
        ax_row[3],
        ori_hash.numpy().reshape(8, 8),
        f"Original\n{ori_hex[:8]}..."
    )
    draw_hash_grid(
        ax_row[4],
        adv_hash.numpy().reshape(8, 8),
        flip_mask=flip_mask,
        title=f"Adversarial\n{adv_hex[:8]}..."
    )
    draw_hash_grid(
        ax_row[5],
        (ori_hash != adv_hash).float().numpy().reshape(8, 8),
        flip_mask=flip_mask,
        title=f"XOR: {bits_flipped}/64 = {d_norm:.3f}\nT=0.10 ✓"
    )

print("Render function defined.")

## Figure 1 — Five diverse images (minimum-cost evasion, 8 bits each)

In [ ]:
# Pick 5 diverse examples: fast (step≤4), medium, slow(er), best SSIM, worst SSIM
random.seed(42)

fastest    = confirmed_pairs[0]   # lowest step count
best_ssim  = max(confirmed_pairs, key=lambda x: x['ssim'])
worst_ssim = min(confirmed_pairs, key=lambda x: x['ssim'])
best_linf  = min(confirmed_pairs, key=lambda x: x['linf'])
mid        = confirmed_pairs[len(confirmed_pairs) // 2]

selected = [fastest, best_ssim, best_linf, mid, worst_ssim]
labels   = ['Fastest (min steps)', 'Best SSIM', 'Best L∞ (most stealthy)',
            'Median difficulty', 'Worst SSIM']

print("Selected examples:")
for pair, label in zip(selected, labels):
    print(f"  [{label}]")
    print(f"    {pair['image_id']}")
    print(f"    steps={pair['step']}  SSIM={pair['ssim']:.4f}  L∞={pair['linf']:.4f}")

In [ ]:
n = len(selected)
fig, axes = plt.subplots(n, 6, figsize=(17, 3.2 * n))

col_headers = [
    'Original', 'Adversarial', 'Difference ×10',
    'Original hash', 'Adversarial hash', 'Flipped bits (XOR)'
]

# Column headers on top row only
for ax, hdr in zip(axes[0], col_headers):
    ax.set_title(hdr, fontsize=10, fontweight='bold', pad=6)

for idx, (pair, label) in enumerate(zip(selected, labels)):
    render_row(axes[idx], pair)
    axes[idx][0].set_ylabel(
        label, fontsize=8, rotation=0, labelpad=70, va='center'
    )

plt.suptitle(
    'pHash Evasion — Image and Hash Visualization\n'
    'T = 0.10  |  8 bits flipped (d_norm = 0.125)  |  '
    'Blue = bit 1  |  Gray = bit 0  |  Red = flipped bit',
    fontsize=10, y=1.01
)
plt.tight_layout()

out = os.path.join(FIG_DIR, 'phash_attack_visualization_T010_8bit.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", out)

## Figure 2 — Fastest 4 attacks side by side (step count ≤ 4)

In [ ]:
# Images that evaded in 4 steps or fewer
ultra_fast = [p for p in confirmed_pairs if p['step'] <= 4]
print(f"Images that evaded in ≤4 steps: {len(ultra_fast)}")

if len(ultra_fast) >= 4:
    show = ultra_fast[:4]
    fig2, axes2 = plt.subplots(4, 6, figsize=(17, 13))
    for ax, hdr in zip(axes2[0], col_headers):
        ax.set_title(hdr, fontsize=10, fontweight='bold', pad=6)
    for idx, pair in enumerate(show):
        render_row(axes2[idx], pair)
        axes2[idx][0].set_ylabel(
            f"{pair['step']} steps", fontsize=9, rotation=0, labelpad=55, va='center'
        )
    plt.suptitle(
        'pHash Evasion — Fastest attacks (≤4 steps, 8 bits flipped, T=0.10)',
        fontsize=10, y=1.01
    )
    plt.tight_layout()
    out2 = os.path.join(FIG_DIR, 'phash_attack_visualization_ultrafast.png')
    plt.savefig(out2, dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved:", out2)
else:
    print("Fewer than 4 ultra-fast examples — adjust step threshold")

## Single-image hash trace — full bit-by-bit breakdown

In [ ]:
# Use the best_linf example — most stealthy, clearest thesis example
pair = best_linf
print(f"Image: {pair['image_id']}")
print(f"Steps: {pair['step']}  Time: {pair['elapsed_ms']:.0f}ms")
print(f"SSIM: {pair['ssim']:.4f}  L∞: {pair['linf']:.4f}  L2: {pair['l2']:.4f}")

ori_t = load_and_preprocess_img(pair['ori_path'], device, resize=True)
adv_t = load_and_preprocess_img(pair['adv_path'], device, resize=True)
ori_hash, ori_hex, ori_med = compute_phash_hard(ori_t)
adv_hash, adv_hex, _       = compute_phash_hard(adv_t)

ori_bits = ori_hash.int().tolist()
adv_bits = adv_hash.int().tolist()
flipped  = [(i, ori_bits[i], adv_bits[i]) for i in range(64) if ori_bits[i] != adv_bits[i]]

print(f"\nOriginal hash  : {ori_hex}")
print(f"Adversarial    : {adv_hex}")
print(f"Bits flipped   : {len(flipped)} / 64  (d_norm = {len(flipped)/64:.4f})")
print(f"DCT median     : {ori_med:.6f}")
print()
print("Flipped positions:")
print(f"{'Pos':>4} | {'Row':>4} | {'Col':>4} | {'Orig':>5} | {'Adv':>5}")
print("-" * 35)
for pos, orig, adv in flipped:
    print(f"{pos:>4} | {pos//8:>4} | {pos%8:>4} | {orig:>5} | {adv:>5}")

print()
print("Hash rows (original):")
for r in range(8):
    row = ori_bits[r*8:(r+1)*8]
    flip_row = [ori_bits[r*8+c] != adv_bits[r*8+c] for c in range(8)]
    annotated = [f"[{b}]" if f else f" {b} " for b, f in zip(row, flip_row)]
    print(f"  row {r}: {''.join(annotated)}  ← [x] = flipped")

## DCT median threshold distribution across 500 images

In [ ]:
import pandas as pd
from utils.phash_torch import compute_phash_soft

all_images = sorted(glob.glob(os.path.join(ORI_DIR, '*')))
medians = []

for img_path in all_images:
    try:
        t = load_and_preprocess_img(img_path, device, resize=True)
        _, _, m = compute_phash_hard(t)
        medians.append(float(m))
    except Exception:
        pass

medians = np.array(medians)
print(f"Medians computed for {len(medians)} images")
print(f"  Median of medians : {np.median(medians):.4f}")
print(f"  Std               : {np.std(medians):.4f}")
print(f"  Min / Max         : {medians.min():.4f} / {medians.max():.4f}")
print()
print("Scientific interpretation:")
print("  The per-image median is approximately zero for natural images (ImageNette).")
print("  This means pHash is essentially encoding the sign pattern of low-frequency")
print("  DCT coefficients. The attack works by flipping enough signs.")
print("  The narrow std (0.049) confirms that ImageNette is representative of")
print("  natural image DCT statistics — supporting generalizability of results.")
print("  A dataset with systematically different DCT distributions (medical images,")
print("  satellite imagery) would have a shifted median, but the per-image frozen-median")
print("  attack strategy self-calibrates and would still succeed.")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(medians, bins=40, color='#378ADD', edgecolor='white', linewidth=0.3)
ax.axvline(np.median(medians), color='red', lw=1.5, linestyle='--',
           label=f'Median = {np.median(medians):.4f}')
ax.axvline(0, color='#888', lw=0.8, linestyle=':', label='Zero')
ax.set_xlabel('Per-image DCT median threshold', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(
    'Distribution of pHash median thresholds across 500 ImageNette images\n'
    f'std = {np.std(medians):.4f} — attack self-calibrates per image',
    fontsize=10
)
ax.legend(fontsize=10)
plt.tight_layout()
out = os.path.join(FIG_DIR, 'phash_median_distribution.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", out)

## Summary stats for 8-bit evasion subset

In [ ]:
steps_8 = [p['step']       for p in confirmed_pairs]
times_8 = [p['elapsed_ms'] for p in confirmed_pairs]
ssims_8 = [p['ssim']       for p in confirmed_pairs]
linfs_8 = [p['linf']       for p in confirmed_pairs]

print("=" * 55)
print(f"  8-bit evasion subset  ({len(confirmed_pairs)} images)")
print("=" * 55)
print(f"  Median steps : {np.median(steps_8):.0f}")
print(f"  P95 steps    : {np.percentile(steps_8, 95):.0f}")
print(f"  Min steps    : {min(steps_8)}")
print(f"  Median time  : {np.median(times_8):.0f} ms")
print(f"  P95 time     : {np.percentile(times_8, 95):.0f} ms")
print(f"  Median SSIM  : {np.median(ssims_8):.4f}")
print(f"  Min SSIM     : {min(ssims_8):.4f}")
print(f"  Median L∞    : {np.median(linfs_8):.4f}")
print(f"  Max L∞       : {max(linfs_8):.4f}")
print()
print(f"  Note: all 383 images flipped exactly 8/64 bits (d_norm=0.125)")
print(f"  This is the minimum-cost evasion — the attack stopped as soon")
print(f"  as the threshold was crossed, not a step further.")